# P3 · Notebook 02b: Semantic check, router and quality-gated cascade (GPU)

**The main idea of our project:** most prompts don't need the expensive setting. So we **start cheap, check the picture, and spend more compute only when it fails**.

```
prompt ─► router picks a starting level ─► SANA-Sprint draws ─► GATE: visual check (classical CV) + semantic check (detector + CLIP)
                                                ▲                         │ pass → done
                                                └── next level (more steps, bigger, NEW seed) ◄── fail
```

**What this notebook does:**
1. **Validates the measuring tools** on real COCO photos with ground truth: can our checkers tell a true checklist from a false one?
2. **Generates every prompt at every level**: training prompts and test prompts, kept strictly apart.
3. **Labels from experiments:** each *training* prompt's label is the **cheapest level whose picture passes the gate**.
4. **Trains the router** (prompt → starting level) and tests it on held-out prompts.
5. **Replays 6 strategies on the test prompts** (fixed levels, cascade, router, router + cascade) and compares speed against quality.
6. **Tests the original model** (the 20-step teacher, notebook 01's baseline) on the **same** test prompts with the **same** judge. That's what lets us say "better than / as good as the original".

**Two separate judges, so the cascade never grades its own homework:**

| | Detector | CLIP | Used for |
|---|---|---|---|
| **Gate** (fast) | Mask R-CNN v2 (torchvision) | ViT-B/16 | deciding *during* the cascade |
| **Judge** (independent) | **Mask2Former Swin-S**, the kind GenEval uses | **ViT-L/14**, same as notebook 01 | **scoring** every strategy |

All structured scores are **"GenEval-style"** (our re-implementation of GenEval's rules), **not** official GenEval numbers.

**Before running:**
- Runtime → **T4 GPU**.
- `RUNGS` are already set from notebook 01's T4 results: 768px → 1024px → 1024px, with a new seed at every level.
- QUICK mode ≈ 1–1.5 hours (including ~15–20 min for the original teacher). FULL mode ≈ 4–5 hours.
- Progress is saved to Drive. If Colab disconnects, **Run all** again.

## 1 · Settings

In [ ]:
import os
QUICK = True
N_TRAIN_STRUCT = 120 if QUICK else 400    # our own GenEval-like prompts (same templates, none of GenEval's own)
N_TRAIN_COCO = 80 if QUICK else 300       # COCO *train* captions
N_TEST_GENEVAL = 120 if QUICK else 553    # official GenEval prompts (QUICK: equal share per task)
N_TEST_COCO = 80 if QUICK else 300        # COCO *val* captions
N_CHECKER_PHOTOS = 100 if QUICK else 300  # real COCO photos for validating the checkers
if os.environ.get("P3_TINY"):             # offline smoke test only
    N_TRAIN_STRUCT, N_TRAIN_COCO, N_TEST_GENEVAL, N_TEST_COCO, N_CHECKER_PHOTOS = 8, 4, 8, 4, 6

MODEL_REPO = "Efficient-Large-Model/Sana_Sprint_0.6B_1024px_diffusers"
TE_QUANT = None        # plain float16 = the fastest pipeline (S8/S1 timings); the app may use M3 to save memory
# The cascade's levels, cheapest first, chosen from notebook 01's T4 results:
#   2 steps @768px 0.96 s (CLIP 26.6) | 2 steps @1024px 1.23 s (CLIP 27.1); baseline teacher 5.54 s (CLIP 27.0)
#   - 512px is NOT used: the model was trained at 1024px and 512px pictures look visibly broken (hard-prompt grid)
#   - steps aren't the lever (1 step wasn't faster than 2, 4 steps didn't raise CLIP), so levels change size + seed
#   - every level uses a NEW seed, so a failed composition gets a genuinely new picture
RUNGS = [
    {"name": "R0", "steps": 2, "res": 768},
    {"name": "R1", "steps": 2, "res": 1024},
    {"name": "R2", "steps": 2, "res": 1024},
]
GATE_CLIP = "openai/clip-vit-base-patch16"
JUDGE_CLIP = "openai/clip-vit-large-patch14"
CLIP_MIN_PERCENTILE = 20   # gate: CLIP below the 20th percentile of top-level training pictures = "weak match"
RUN_TEACHER = True         # also test the ORIGINAL model (teacher, 20 steps) on the same test prompts (+15-20 min QUICK)
TEACHER_REPO = "Efficient-Large-Model/Sana_Sprint_0.6B_1024px_teacher_diffusers"
SEED = 0
SAVE_TO_DRIVE = True

## 2 · Install (~1–2 min)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "diffusers", "transformers", "accelerate",
                "bitsandbytes", "scikit-learn", "tabulate", "sentencepiece", "protobuf"], check=True)
print("Libraries installed.")

## 3 · Our modules
Two cells write `p3_classical.py` (visual check, from notebook 02a) and `p3_semantic.py` (prompt reader + GenEval-style checker). They're the same files as `src/` in the repo, and they're copied to Drive (`P3_T2I/shared/`) for the app.

In [ ]:
import gc, json, time, math, random, zipfile, shutil, importlib, textwrap, types, zlib, urllib.request
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter
from IPython.display import display
try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **k: x
import p3_classical as pcv
import p3_semantic as psem
importlib.reload(pcv)
importlib.reload(psem)

MOCK = os.environ.get("P3_MOCK_GENERATOR") == "1"     # offline testing only: a fake generator, no GPU
DEV = "cuda" if torch.cuda.is_available() else "cpu"
assert DEV == "cuda" or MOCK, "No GPU found: Runtime > Change runtime type > T4 GPU, then Run all."
DTYPE = torch.bfloat16 if DEV == "cuda" and torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
GEN_BS = 4 if DEV == "cuda" and torch.cuda.get_device_properties(0).total_memory > 20e9 else 2
GATE_CLIP = os.environ.get("P3_GATE_CLIP", GATE_CLIP)
JUDGE_CLIP = os.environ.get("P3_JUDGE_CLIP", JUDGE_CLIP)
print("Device:", torch.cuda.get_device_name(0) if DEV == "cuda" else "CPU (mock mode)", "| dtype:", DTYPE)

def free_gpu():
    gc.collect()
    if DEV == "cuda":
        torch.cuda.empty_cache()
    try:
        import ctypes
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass

def sync_time():
    if DEV == "cuda":
        torch.cuda.synchronize()
    return time.perf_counter()

OUT = Path(os.environ.get("P3_OUT", "/content/P3_02b_out"))
SHARED = OUT / "shared"
WORK = Path(os.environ.get("P3_WORK", "/content/p3_02b_work"))
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT = Path("/content/drive/MyDrive/P3_T2I/02b_router_cascade")
        SHARED = Path("/content/drive/MyDrive/P3_T2I/shared")
    except Exception as e:
        print("Google Drive not available, saving locally:", e)
MODE_TAG = "quick" if QUICK else "full"
FIG = OUT / f"figures_{MODE_TAG}"
IMG_DIR = WORK / f"images_{MODE_TAG}"
for d in (OUT, SHARED, WORK, FIG, IMG_DIR):
    d.mkdir(parents=True, exist_ok=True)
for f in ("p3_classical.py", "p3_semantic.py"):
    shutil.copy(f, SHARED / f)
MATRIX_CSV = OUT / f"matrix_{MODE_TAG}.csv"
MATRIX_COLS = ["pid", "split", "source", "tag", "rung", "rung_idx", "seed", "vis_flags", "vis_ok", "vis_ms",
               "gate_clip", "gate_struct", "gate_ms", "gate_reasons", "judge_clip", "judge_struct", "judge_reasons"]

def read_matrix():
    """Read the matrix CSV, repairing older files whose header has only the first 14 columns
    (test rows carry 3 extra judge columns at the end). Rewrites the file with the full header."""
    import csv
    if not MATRIX_CSV.exists():
        return pd.DataFrame(columns=MATRIX_COLS)
    with open(MATRIX_CSV, newline="", encoding="utf-8") as f:
        rows = list(csv.reader(f))
    header, body = rows[0], rows[1:]
    if header != MATRIX_COLS:
        assert header == MATRIX_COLS[:len(header)], f"unexpected matrix header: {header}"
        fixed = [r + [""] * (len(MATRIX_COLS) - len(r)) for r in body if len(r) in (len(header), len(MATRIX_COLS))]
        print(f"Repaired matrix CSV: {len(fixed)} rows kept, {len(body) - len(fixed)} broken rows dropped (they will be regenerated)")
        with open(MATRIX_CSV, "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f); w.writerow(MATRIX_COLS); w.writerows(fixed)
    return pd.read_csv(MATRIX_CSV)
print("Results folder:", OUT)

## 4 · Prompt sets (train and test kept strictly apart)

| Set | Source | Used for |
|---|---|---|
| train / struct | **our own** GenEval-style prompts (same templates, random new combinations, none equal to a GenEval prompt) | router labels |
| train / coco | COCO **train** captions | router labels + gate threshold |
| test / geneval | **official GenEval prompts** | scoring all strategies |
| test / coco | COCO **val** captions | scoring all strategies |

In [ ]:
ANN_ZIP = Path(os.environ.get("P3_COCO_ANN", str(WORK / "annotations_trainval2017.zip")))
if not ANN_ZIP.exists():
    print("Downloading COCO annotations (~250 MB, once)...")
    urllib.request.urlretrieve("http://images.cocodataset.org/annotations/annotations_trainval2017.zip", ANN_ZIP)

GENEVAL_FILE = WORK / "geneval_metadata.jsonl"
if not GENEVAL_FILE.exists():
    urllib.request.urlretrieve("https://raw.githubusercontent.com/djghosh13/geneval/main/prompts/evaluation_metadata.jsonl",
                               GENEVAL_FILE)
geneval = [json.loads(line) for line in open(GENEVAL_FILE, encoding="utf-8")]

# Test: official GenEval prompts (QUICK: the same number from each of the 6 tasks)
rng = random.Random(SEED)
by_tag = defaultdict(list)
for g in geneval:
    by_tag[g["tag"]].append(g)
if N_TEST_GENEVAL >= len(geneval):
    test_geneval = geneval
else:
    per = max(1, N_TEST_GENEVAL // len(by_tag))
    test_geneval = [x for t in sorted(by_tag) for x in rng.sample(by_tag[t], min(per, len(by_tag[t])))]

# Train: our own prompts from GenEval's templates (different random combinations)
def article(n):
    return ("an " if n[0] in "aeiou" else "a ") + n

def plural(n):
    return n + ("es" if n.endswith("s") else "s")      # GenEval's own plural rule

NUMS = ["zero", "one", "two", "three", "four"]

def make_struct(r):
    tag = r.choice(["single_object", "two_object", "counting", "colors", "position", "color_attr"])
    a, b = r.sample(psem.CLASSES, 2)
    ca, cb = r.sample(psem.COLOURS, 2)
    if tag == "single_object":
        return dict(tag=tag, include=[{"class": a, "count": 1}], prompt=f"a photo of {article(a)}")
    if tag == "two_object":
        return dict(tag=tag, include=[{"class": a, "count": 1}, {"class": b, "count": 1}],
                    prompt=f"a photo of {article(a)} and {article(b)}")
    if tag == "counting":
        n = r.randint(2, 4)
        return dict(tag=tag, include=[{"class": a, "count": n}], exclude=[{"class": a, "count": n + 1}],
                    prompt=f"a photo of {NUMS[n]} {plural(a)}")
    if tag == "colors":
        return dict(tag=tag, include=[{"class": a, "count": 1, "color": ca}], prompt=f"a photo of {article(ca)} {a}")
    if tag == "position":
        rel = r.choice(["left of", "right of", "above", "below"])
        return dict(tag=tag, include=[{"class": b, "count": 1}, {"class": a, "count": 1, "position": [rel, 0]}],
                    prompt=f"a photo of {article(a)} {rel} {article(b)}")
    return dict(tag=tag, include=[{"class": a, "count": 1, "color": ca}, {"class": b, "count": 1, "color": cb}],
                prompt=f"a photo of {article(ca)} {a} and {article(cb)} {b}")

rng_train = random.Random(SEED + 2024)
seen = {g["prompt"] for g in geneval}
train_struct = []
while len(train_struct) < N_TRAIN_STRUCT:
    s = make_struct(rng_train)
    if s["prompt"] not in seen:
        seen.add(s["prompt"])
        train_struct.append(s)

# COCO captions: train split for training, val split for testing (different images)
with zipfile.ZipFile(ANN_ZIP) as z:
    cap_train = json.loads(z.read("annotations/captions_train2017.json"))["annotations"]
    cap_val = json.loads(z.read("annotations/captions_val2017.json"))["annotations"]

def first_captions(anns, n, seed):
    first = {}
    for a in anns:
        first.setdefault(a["image_id"], " ".join(a["caption"].split()))
    ids = sorted(first)
    random.Random(seed).shuffle(ids)
    return [first[i] for i in ids[:n]]

coco_train = first_captions(cap_train, N_TRAIN_COCO, SEED)
coco_test = first_captions(cap_val, N_TEST_COCO, SEED + 1)
del cap_train, cap_val

def spec_of(d):
    return {k: d[k] for k in ("include", "exclude") if k in d}

PROMPTS = ([dict(split="train", source="struct", tag=s["tag"], prompt=s["prompt"], spec=spec_of(s)) for s in train_struct]
           + [dict(split="train", source="coco", tag="coco", prompt=p, spec=psem.parse_prompt(p)) for p in coco_train]
           + [dict(split="test", source="geneval", tag=g["tag"], prompt=g["prompt"], spec=spec_of(g)) for g in test_geneval]
           + [dict(split="test", source="coco", tag="coco", prompt=p, spec=psem.parse_prompt(p)) for p in coco_test])
for i, p in enumerate(PROMPTS):
    p["pid"] = i
json.dump(PROMPTS, open(OUT / f"prompts_{MODE_TAG}.json", "w"), indent=1)

# Our prompt reader must reproduce the template checklists exactly (it's what the app uses for typed prompts)
def canon(spec):
    inc = spec.get("include", [])
    item = lambda r: (r["class"], r["count"], r.get("color"), (r["position"][0], inc[r["position"][1]]["class"]) if "position" in r else None)
    return (sorted(map(item, inc)), sorted((r["class"], r["count"]) for r in spec.get("exclude", [])))
templ = [p for p in PROMPTS if p["source"] in ("struct", "geneval")]
agree = np.mean([psem.parse_prompt(p["prompt"]) is not None and canon(psem.parse_prompt(p["prompt"])) == canon(p["spec"]) for p in templ])
print(pd.Series(Counter((p["split"], p["source"]) for p in PROMPTS)).to_string())
print(f"Prompt reader agrees with the true checklist on {agree:.1%} of {len(templ)} structured prompts")
print(f"COCO captions with something checkable: {np.mean([p['spec'] is not None for p in PROMPTS if p['source'] == 'coco']):.0%}")

## 5 · Load both checkers (the gate and the independent judge)
Mask2Former may print a load report saying `swin.layernorm` is "MISSING / newly initialized". That's expected: the detector doesn't use that final backbone layer. Section 6 checks on real photos that the detector works.

In [ ]:
gate = psem.SemanticChecker(device=DEV, clip_name=GATE_CLIP, detector="maskrcnn")
free_gpu()
judge = psem.SemanticChecker(device=DEV, clip_name=JUDGE_CLIP, detector="mask2former")
free_gpu()
if DEV == "cuda":
    print(f"GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 6 · Can we trust the measuring tools? (checker validation on real COCO photos)
COCO's annotations say which objects are really in each photo. From them we build **true** checklists (should pass) and **false** ones (absent object, wrong count, flipped position: should fail). This runs *before* the generator is loaded, because the annotation file needs a lot of RAM.

In [ ]:
with zipfile.ZipFile(ANN_ZIP) as z:
    inst = json.loads(z.read("annotations/instances_val2017.json"))
cat_name = {c["id"]: {"mouse": "computer mouse", "remote": "tv remote", "keyboard": "computer keyboard"}.get(c["name"], c["name"])
            for c in inst["categories"]}
img_meta = {im["id"]: im for im in inst["images"]}
anns_by_img = defaultdict(list)
for a in inst["annotations"]:
    if not a["iscrowd"]:
        anns_by_img[a["image_id"]].append(a)
del inst
PHOTO_DIR = WORK / "coco_val_photos"
PHOTO_DIR.mkdir(exist_ok=True)
photo_ids = sorted(anns_by_img)
random.Random(SEED + 7).shuffle(photo_ids)
photo_ids = photo_ids[:N_CHECKER_PHOTOS]
FLIP = {"left of": "right of", "right of": "left of", "above": "below", "below": "above"}

def gt_specs(iid, r):
    im = img_meta[iid]
    anns = anns_by_img[iid]
    big = [a for a in anns if a["area"] >= 0.01 * im["width"] * im["height"]]
    n_all = Counter(cat_name[a["category_id"]] for a in anns)
    n_big = Counter(cat_name[a["category_id"]] for a in big)
    specs = [("presence true", True, {"include": [{"class": c, "count": 1}]}) for c in n_big]
    absent = [c for c in psem.CLASSES if c not in n_all]
    specs.append(("presence false", False, {"include": [{"class": r.choice(absent), "count": 1}]}))
    for c, n in n_big.items():
        if 2 <= n <= 4 and n_all[c] == n:
            specs.append(("count true", True, {"include": [{"class": c, "count": n}], "exclude": [{"class": c, "count": n + 1}]}))
            specs.append(("count false", False, {"include": [{"class": c, "count": n + 1}], "exclude": [{"class": c, "count": n + 2}]}))
    singles = [a for a in big if n_all[cat_name[a["category_id"]]] == 1]
    box = lambda q: [q["bbox"][0], q["bbox"][1], q["bbox"][0] + q["bbox"][2], q["bbox"][1] + q["bbox"][3]]
    for a, b in [(x, y) for x in singles for y in singles if x is not y][:2]:
        rels = psem.relative_position(box(a), box(b))
        if len(rels) == 1:
            rel = rels.pop()
            ca, cb = cat_name[a["category_id"]], cat_name[b["category_id"]]
            for kind, rr, exp in (("position true", rel, True), ("position false", FLIP[rel], False)):
                specs.append((kind, exp, {"include": [{"class": cb, "count": 1}, {"class": ca, "count": 1, "position": [rr, 0]}]}))
    return specs

val_rows = []
r = random.Random(SEED + 8)
for iid in tqdm(photo_ids, desc="Validating checkers"):
    f = PHOTO_DIR / img_meta[iid]["file_name"]
    if not f.exists():
        urllib.request.urlretrieve(f"http://images.cocodataset.org/val2017/{img_meta[iid]['file_name']}", f)
    photo = Image.open(f).convert("RGB")
    specs = gt_specs(iid, r)
    for name, checker in (("gate", gate), ("judge", judge)):
        dets = checker.detect(photo)
        for kind, expected, spec in specs:
            val_rows.append(dict(checker=name, kind=kind, correct=checker.check(photo, spec, detections=dets)["passed"] == expected))
val = pd.DataFrame(val_rows)
val_table = val.pivot_table(index="kind", columns="checker", values="correct", aggfunc="mean").round(3)
val_table["n"] = val[val.checker == "gate"].groupby("kind").size()
display(val_table)
val_table.to_csv(OUT / f"checker_validation_{MODE_TAG}.csv")
print("Read: share of correct verdicts. 'true' rows = should pass, 'false' rows = should fail.")

## 7 · Load the generator; speed of each level and of the gate (single picture, what a user waits for)

In [ ]:
class MockPipe:
    """Offline testing only: returns real photos instead of generating (the cheapest level is sometimes blurry)."""
    def __init__(self):
        self.files = sorted(Path(os.environ["P3_MOCK_IMAGES"]).glob("*.jpg"))
    def __call__(self, prompt, generator, height, width, num_inference_steps, **kw):
        imgs = []
        for p, g in zip(prompt, generator):
            k = (zlib.crc32(p.encode()) + int(g.initial_seed())) % len(self.files)
            im = Image.open(self.files[k]).convert("RGB").resize((width, height))
            if width < 1024 and k % 3 == 0:
                im = im.filter(ImageFilter.GaussianBlur(3))
            imgs.append(im)
        return types.SimpleNamespace(images=imgs)
    def set_progress_bar_config(self, **kw):
        pass

def load_generator():
    if MOCK:
        return MockPipe()
    from diffusers import SanaSprintPipeline, PipelineQuantizationConfig
    kwargs = {"torch_dtype": DTYPE}
    if TE_QUANT:
        from transformers import BitsAndBytesConfig
        q = (BitsAndBytesConfig(load_in_8bit=True) if TE_QUANT == "8bit" else
             BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE))
        kwargs["quantization_config"] = PipelineQuantizationConfig(quant_mapping={"text_encoder": q})
    try:
        pipe = SanaSprintPipeline.from_pretrained(MODEL_REPO, device_map="cuda", **kwargs)   # straight to GPU, saves RAM
    except Exception as e:
        print("device_map='cuda' not supported, loading the normal way:", str(e)[:100])
        free_gpu()
        pipe = SanaSprintPipeline.from_pretrained(MODEL_REPO, **kwargs)
        pipe.to(DEV)
    pipe.set_progress_bar_config(disable=True)
    return pipe

def call_kwargs(rung):
    kw = {"height": rung["res"], "width": rung["res"], "num_inference_steps": rung["steps"],
          "intermediate_timesteps": 1.3 if rung["steps"] == 2 else None}
    if rung["res"] != 1024:
        kw["use_resolution_binning"] = False
    return kw

@torch.inference_mode()
def generate(pipe, rung, prompts, seeds):
    gens = [torch.Generator("cpu").manual_seed(int(s)) for s in seeds]
    return pipe(prompt=list(prompts), generator=gens, **call_kwargs(rung)).images

pipe = load_generator()
free_gpu()
if DEV == "cuda":
    print(f"GPU memory in use with generator + both checkers: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
def time_rung(rung, n=8):
    for i in range(2):
        generate(pipe, rung, [PROMPTS[i]["prompt"]], [SEED])
    times = []
    for i in range(n):
        t0 = sync_time()
        generate(pipe, rung, [PROMPTS[i % len(PROMPTS)]["prompt"]], [SEED + i])
        times.append(sync_time() - t0)
    return float(np.median(times))

RUNG_LAT = {r["name"]: time_rung(r) for r in RUNGS}
sample = generate(pipe, RUNGS[-1], [PROMPTS[0]["prompt"]], [SEED])[0]
t_vis = []
t_sem = []
for p in PROMPTS[:6]:
    t0 = time.perf_counter()
    pcv.assess(np.asarray(sample), p["prompt"])
    t_vis.append(time.perf_counter() - t0)
    t_sem.append(gate.gate(sample, p["prompt"], spec=p["spec"])["ms"] / 1000)
GATE_LAT = float(np.median(t_vis) + np.median(t_sem))
speed = pd.DataFrame({"seconds": {**{f"generate {k} ({r['steps']} step, {r['res']}px)": v for (k, v), r in zip(RUNG_LAT.items(), RUNGS)},
                                  "gate check (visual + semantic)": GATE_LAT}})
display(speed.round(3))
speed.to_csv(OUT / f"speed_{MODE_TAG}.csv")

## 8 · The experiment: every prompt at every level (the long part, resumable)
Each level uses a **different seed**, so going up a level gives a genuinely new picture, not just a sharper copy of the same one. The **gate** scores everything. The **judge** scores the test prompts only.

In [ ]:
done = set()
if MATRIX_CSV.exists():
    old = read_matrix()
    done = set(zip(old["pid"], old["rung"]))
    print(f"Resuming: {len(done)} pictures already done")
buffer = []

def flush():
    global buffer
    if buffer:
        pd.DataFrame(buffer).reindex(columns=MATRIX_COLS).to_csv(MATRIX_CSV, mode="a", header=not MATRIX_CSV.exists(), index=False)
        buffer = []

t_start = time.time()
for ri, rung in enumerate(RUNGS):
    todo = [p for p in PROMPTS if (p["pid"], rung["name"]) not in done]
    for k in tqdm(range(0, len(todo), GEN_BS), desc=f"{rung['name']} ({rung['steps']} step, {rung['res']}px)"):
        batch = todo[k:k + GEN_BS]
        seeds = [SEED + 100000 * ri + p["pid"] for p in batch]
        images = generate(pipe, rung, [p["prompt"] for p in batch], seeds)
        for p, im, seed in zip(batch, images, seeds):
            t0 = time.perf_counter()
            vis = pcv.assess(np.asarray(im), p["prompt"])
            vis_ms = (time.perf_counter() - t0) * 1000
            g = gate.gate(im, p["prompt"], spec=p["spec"])
            row = dict(pid=p["pid"], split=p["split"], source=p["source"], tag=p["tag"], rung=rung["name"], rung_idx=ri,
                       seed=seed, vis_flags=",".join(vis["flags"]), vis_ok=not any(f in pcv.UNFIXABLE for f in vis["flags"]),
                       vis_ms=vis_ms, gate_clip=g["clip"], gate_struct=None if g["check"] is None else g["check"]["passed"],
                       gate_ms=g["ms"], gate_reasons="; ".join(g["reasons"])[:300])
            if p["split"] == "test":
                j = judge.gate(im, p["prompt"], spec=p["spec"])
                row.update(judge_clip=j["clip"], judge_struct=None if j["check"] is None else j["check"]["passed"],
                           judge_reasons="; ".join(j["reasons"])[:300])
                small = im if im.width <= 768 else im.resize((768, 768), Image.LANCZOS)   # keep the cheapest level at native size
                small.save(IMG_DIR / f"{p['pid']}_{rung['name']}.jpg", quality=92)
            buffer.append(row)
        if len(buffer) >= 40:
            flush()
    flush()
print(f"Experiment finished in {(time.time() - t_start) / 60:.1f} min")

## 8b · The original model on the same test prompts (for a fair "vs original" comparison)
This runs the SANA-Sprint **teacher** (20 steps + guidance, notebook 01's baseline) on exactly the same test prompts, scored by the same judge. The fast model is unloaded first, because both won't fit on a T4 together. QUICK adds ≈ 15–20 min. It's resumable.

In [ ]:
TEACHER_CSV = OUT / f"teacher_{MODE_TAG}.csv"
TEACHER_LAT_FILE = OUT / f"teacher_latency_{MODE_TAG}.json"
test_prompts = [p for p in PROMPTS if p["split"] == "test"]
t_done = set(pd.read_csv(TEACHER_CSV)["pid"]) if TEACHER_CSV.exists() else set()
t_todo = [p for p in test_prompts if p["pid"] not in t_done]

def load_teacher():
    if MOCK:
        return MockPipe()
    from diffusers import SanaPipeline
    try:
        t = SanaPipeline.from_pretrained(TEACHER_REPO, torch_dtype=DTYPE, device_map="cuda")
    except Exception as e:
        print("device_map='cuda' not supported, loading the normal way:", str(e)[:100])
        free_gpu()
        t = SanaPipeline.from_pretrained(TEACHER_REPO, torch_dtype=DTYPE)
        t.to(DEV)
    t.set_progress_bar_config(disable=True)
    return t

@torch.inference_mode()
def teacher_generate(t, prompts, seeds):
    gens = [torch.Generator("cpu").manual_seed(int(s)) for s in seeds]
    return t(prompt=list(prompts), generator=gens, num_inference_steps=20, guidance_scale=4.5,
             height=1024, width=1024).images

if RUN_TEACHER and (t_todo or not TEACHER_LAT_FILE.exists()):
    pipe = None                       # unload the fast model: both don't fit on a T4
    free_gpu()
    teacher = load_teacher()
    if not TEACHER_LAT_FILE.exists():
        teacher_generate(teacher, [PROMPTS[0]["prompt"]], [SEED])          # warm-up
        times = []
        for i in range(3):
            t0 = sync_time()
            teacher_generate(teacher, [test_prompts[i % len(test_prompts)]["prompt"]], [SEED + i])
            times.append(sync_time() - t0)
        json.dump({"seconds": float(np.median(times))}, open(TEACHER_LAT_FILE, "w"))
    rows = []
    for p in tqdm(t_todo, desc="Original teacher (20 steps)"):
        im = teacher_generate(teacher, [p["prompt"]], [SEED + 900000 + p["pid"]])[0]
        vis = pcv.assess(np.asarray(im), p["prompt"])
        j = judge.gate(im, p["prompt"], spec=p["spec"])
        rows.append(dict(pid=p["pid"], source=p["source"], tag=p["tag"], judge_clip=j["clip"],
                         judge_struct=None if j["check"] is None else j["check"]["passed"],
                         judge_reasons="; ".join(j["reasons"])[:300], vis_flags=",".join(vis["flags"]),
                         vis_ok=not any(f in pcv.UNFIXABLE for f in vis["flags"])))
        im.resize((768, 768), Image.LANCZOS).save(IMG_DIR / f"{p['pid']}_TEACHER.jpg", quality=92)
        if len(rows) >= 10:
            pd.DataFrame(rows).to_csv(TEACHER_CSV, mode="a", header=not TEACHER_CSV.exists(), index=False)
            rows = []
    if rows:
        pd.DataFrame(rows).to_csv(TEACHER_CSV, mode="a", header=not TEACHER_CSV.exists(), index=False)
    teacher = None
    free_gpu()
TEACHER_LAT = json.load(open(TEACHER_LAT_FILE))["seconds"] if TEACHER_LAT_FILE.exists() else None
print("Teacher seconds per picture:", None if TEACHER_LAT is None else round(TEACHER_LAT, 2))

## 9 · Gate rule (set on training prompts only) and router labels

In [ ]:
M = read_matrix()
M = M[M["pid"].isin([p["pid"] for p in PROMPTS])].drop_duplicates(["pid", "rung"], keep="last")
def as_bool(s):
    return s.map({True: True, False: False, "True": True, "False": False})
M["gate_struct_ok"] = as_bool(M["gate_struct"]).fillna(True).astype(bool)     # nothing checkable = no structured veto
M["vis_ok"] = as_bool(M["vis_ok"]).astype(bool)
top = RUNGS[-1]["name"]
CLIP_MIN = float(np.percentile(M[(M.split == "train") & (M.rung == top)]["gate_clip"], CLIP_MIN_PERCENTILE))
M["gate_pass"] = M["vis_ok"] & M["gate_struct_ok"] & (M["gate_clip"] >= CLIP_MIN)
if "judge_struct" in M:
    M["judge_struct_ok"] = as_bool(M["judge_struct"])
print(f"Gate: visual check passes AND structured check passes (if any) AND gate-CLIP >= {CLIP_MIN:.2f}")

pass_table = M.pivot_table(index=["split", "source"], columns="rung", values="gate_pass", aggfunc="mean")[[r["name"] for r in RUNGS]]
display(pass_table.round(3))

# Label = the cheapest level whose picture passes the gate (no level passes -> top level, marked)
P = M.pivot_table(index="pid", columns="rung_idx", values="gate_pass", aggfunc="first").fillna(False)
def first_pass(row):
    for i in range(len(RUNGS)):
        if row.get(i, False):
            return i
    return len(RUNGS) - 1
labels = P.apply(first_pass, axis=1)
none_passed = ~P.any(axis=1)
info = pd.DataFrame([{"pid": p["pid"], "split": p["split"], "source": p["source"], "tag": p["tag"], "prompt": p["prompt"]} for p in PROMPTS]).set_index("pid")
info["label"] = labels
info["none_passed"] = none_passed
print("\nLabel distribution on TRAIN prompts (cheapest level that passes):")
display(info[info.split == "train"].groupby("source")["label"].value_counts(normalize=True).unstack().fillna(0).round(3))
print(f"Train prompts where no level passed the gate: {info[info.split == 'train']['none_passed'].mean():.0%}")

## 10 · The router: prompt → starting level (trained on train, tested on test)
**Inputs:** simple prompt clues (number of objects, counts, colours, positions, text, length) plus a compressed CLIP text embedding. **Model:** a small logistic regression, which is fast, readable and easy to ship to the app as JSON.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

FEATS = list(psem.prompt_features("x").keys())
specs = {p["pid"]: p["spec"] for p in PROMPTS}
X_hand = np.array([[psem.prompt_features(info.loc[pid, "prompt"], specs[pid])[f] for f in FEATS] for pid in info.index], float)
with torch.no_grad():
    E = torch.cat([gate._text_embeds(list(info["prompt"].iloc[i:i + 64])) for i in range(0, len(info), 64)]).float().cpu().numpy()
tr = (info.split == "train").to_numpy()
te = ~tr
scaler = StandardScaler().fit(X_hand[tr])
pca = PCA(n_components=min(16, int(tr.sum()) - 1, E.shape[1]), random_state=SEED).fit(E[tr])
X = np.hstack([scaler.transform(X_hand), pca.transform(E)])
y = info["label"].to_numpy()
router = LogisticRegression(max_iter=3000, class_weight="balanced", C=0.5)
majority = Counter(y[tr]).most_common(1)[0][0]
if len(set(y[tr])) < 2:
    print("Every training prompt has the same label, so there's nothing to learn: the router always predicts", majority)
    cv, pred = np.array([1.0]), np.full(len(y), majority)
else:
    from sklearn.model_selection import KFold
    cv = cross_val_score(router, X[tr], y[tr], scoring="accuracy",
                         cv=KFold(n_splits=min(5, int(tr.sum())), shuffle=True, random_state=SEED))
    router.fit(X[tr], y[tr])
    pred = router.predict(X)
info["router_start"] = pred
print(f"Router accuracy, 5-fold CV on train: {cv.mean():.1%}")
print(f"Router accuracy on TEST labels: {np.mean(pred[te] == y[te]):.1%}  (always-predict-majority baseline: {np.mean(y[te] == majority):.1%})")
cm = pd.crosstab(pd.Series(y[te], name="true cheapest level"), pd.Series(pred[te], name="router says"))
display(cm)

ROUTER_JSON = {
    "features": FEATS, "scaler_mean": scaler.mean_.tolist(), "scaler_scale": scaler.scale_.tolist(),
    "pca_mean": pca.mean_.tolist(), "pca_components": pca.components_.tolist(), "text_encoder": GATE_CLIP,
    "classes": [int(c) for c in getattr(router, "classes_", [majority])],
    "coef": router.coef_.tolist() if hasattr(router, "coef_") else None,
    "intercept": router.intercept_.tolist() if hasattr(router, "intercept_") else None,
}

## 11 · Replay the strategies on TEST prompts (scored by the independent judge)
Because every test prompt was generated at every level, each strategy can be replayed exactly:
- **latency** = time of every level it tried + the gate check after each try
- **quality** = the **judge's** verdict on the picture it finally returns

In [ ]:
T = M[M.split == "test"].set_index(["pid", "rung_idx"]).sort_index()
LAT = [RUNG_LAT[r["name"]] for r in RUNGS]
TOP = len(RUNGS) - 1

def replay(start_of, cascade):
    out = []
    for pid in sorted({i for i, _ in T.index}):
        r = int(start_of(pid))
        cost = 0.0
        while True:
            row = T.loc[(pid, r)]
            cost += LAT[r]
            if not cascade:
                break
            cost += GATE_LAT
            if row["gate_pass"] or r == TOP:
                break
            r += 1
        out.append(dict(pid=pid, final=r, latency=cost, source=row["source"], tag=row["tag"],
                        judge_struct=row["judge_struct_ok"], judge_clip=row["judge_clip"], vis_ok=row["vis_ok"]))
    return pd.DataFrame(out)

STRATEGIES = {f"fixed {r['name']} ({r['steps']} step, {r['res']}px)": (lambda pid, i=i: i, False) for i, r in enumerate(RUNGS)}
STRATEGIES.update({
    "cascade from R0": (lambda pid: 0, True),
    "router only": (lambda pid: info.loc[pid, "router_start"], False),
    "router + cascade": (lambda pid: info.loc[pid, "router_start"], True),
})
results, summary = {}, []
for name, (start_of, cascade) in STRATEGIES.items():
    results[name] = replay(start_of, cascade)
TEACHER_NAME = "ORIGINAL teacher (20 steps, 1024px)"
if TEACHER_CSV.exists() and TEACHER_LAT is not None:
    Tt = pd.read_csv(TEACHER_CSV).drop_duplicates("pid", keep="last")
    Tt = Tt[Tt.pid.isin(results["cascade from R0"].pid)]
    results[TEACHER_NAME] = pd.DataFrame(dict(pid=Tt.pid, final=-1, latency=TEACHER_LAT, source=Tt.source, tag=Tt.tag,
                                              judge_struct=as_bool(Tt.judge_struct), judge_clip=Tt.judge_clip,
                                              vis_ok=as_bool(Tt.vis_ok)))
for name, df in results.items():
    g = df[df.source == "geneval"]
    c = df[df.source == "coco"]
    row = {"strategy": name, "avg_latency_s": df.latency.mean(), "p90_latency_s": df.latency.quantile(0.9),
           "geneval_style_acc": g.judge_struct.astype(float).mean() if len(g) else np.nan,
           "clip_L_coco": c.judge_clip.mean() if len(c) else np.nan, "clip_L_all": df.judge_clip.mean(),
           "visual_ok": df.vis_ok.mean()}
    for i, r in enumerate(RUNGS):
        row[f"ends_at_{r['name']}"] = (df.final == i).mean()
    summary.append(row)
S = pd.DataFrame(summary).set_index("strategy")
display(S.round(3))
S.to_csv(OUT / f"strategies_{MODE_TAG}.csv")

per_task = pd.DataFrame({name: df[df.source == "geneval"].groupby("tag")["judge_struct"].apply(lambda s: s.astype(float).mean())
                         for name, df in results.items()}).T
display(per_task.round(3))
per_task.to_csv(OUT / f"geneval_style_per_task_{MODE_TAG}.csv")

# Oracle (hindsight): the cheapest level whose picture the JUDGE accepts. An upper bound, not a real strategy.
g_test = T[T.source == "geneval"]
oracle = []
for pid in sorted({i for i, _ in g_test.index}):
    ok = [g_test.loc[(pid, i), "judge_struct_ok"] for i in range(len(RUNGS))]
    first = next((i for i, v in enumerate(ok) if v), TOP)
    oracle.append((LAT[first], bool(ok[first])))
print(f"Oracle (hindsight) on GenEval prompts: accuracy {np.mean([o[1] for o in oracle]):.1%} at {np.mean([o[0] for o in oracle]):.2f} s average")
if TEACHER_NAME in S.index:
    t_row, ours = S.loc[TEACHER_NAME], S.loc["router + cascade"]
    print()
    print("OURS (router + cascade) vs ORIGINAL, same prompts and same judge:")
    print(f"  GenEval-style accuracy: {ours.geneval_style_acc:.1%} vs {t_row.geneval_style_acc:.1%}")
    print(f"  CLIP ViT-L/14 (all):    {ours.clip_L_all:.2f} vs {t_row.clip_L_all:.2f}")
    print(f"  average time:           {ours.avg_latency_s:.2f} s vs {t_row.avg_latency_s:.2f} s "
          f"({t_row.avg_latency_s / ours.avg_latency_s:.1f}x faster), incl. our quality checks")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col, label in [(axes[0], "geneval_style_acc", "GenEval-style accuracy (judge)"), (axes[1], "clip_L_all", "CLIP ViT-L/14 score (judge)")]:
    ax.scatter(S["avg_latency_s"], S[col], s=80)
    for name, rr in S.iterrows():
        ax.annotate(name, (rr["avg_latency_s"], rr[col]), fontsize=8, xytext=(5, 4), textcoords="offset points")
    ax.set_xlabel("average seconds per prompt (generation + checks). Left = faster")
    ax.set_ylabel(label + ". Up = better")
    ax.grid(alpha=0.3)
plt.suptitle("Quality vs cost of each strategy on held-out test prompts")
plt.tight_layout()
plt.savefig(FIG / "strategies_quality_vs_cost.png", dpi=140)
plt.show()

## 12 · How often does the fast gate agree with the independent judge?
If the gate often says "pass" when the judge says "fail", the cascade stops too early. This table is the honest check.

In [ ]:
both = T[T["judge_struct_ok"].notna() & T["gate_struct"].notna()].copy()
both["gate_s"] = as_bool(both["gate_struct"]).astype(bool)
both["judge_s"] = both["judge_struct_ok"].astype(bool)
agreement = pd.crosstab(both["gate_s"].map({True: "gate: pass", False: "gate: fail"}),
                        both["judge_s"].map({True: "judge: pass", False: "judge: fail"}), normalize="all").round(3)
display(agreement)
print(f"Agreement: {(both.gate_s == both.judge_s).mean():.1%} of {len(both)} test pictures; "
      f"gate too lenient (pass but judge fails): {((both.gate_s) & (~both.judge_s)).mean():.1%}")

## 13 · Can classical CV replace a more expensive level? (cheapest level + safe polish vs the next level)

In [ ]:
coco_pids = sorted({i for i, _ in T.index if T.loc[(i, 0), "source"] == "coco"})
if len(RUNGS) > 1 and coco_pids:
    raw, polished, changed, prompts_ = [], [], 0, []
    for pid in coco_pids:
        im = Image.open(IMG_DIR / f"{pid}_{RUNGS[0]['name']}.jpg").convert("RGB")
        out, inf = pcv.safe_enhance(np.asarray(im), info.loc[pid, "prompt"])
        changed += inf["accepted"]
        raw.append(im)
        polished.append(Image.fromarray(out))
        prompts_.append(info.loc[pid, "prompt"])
    clip_raw = np.mean(judge.clip_scores(raw, prompts_))
    clip_pol = np.mean(judge.clip_scores(polished, prompts_))
    clip_r1 = T.loc[[(i, 1) for i in coco_pids], "judge_clip"].mean()
    print(f"On {len(coco_pids)} COCO test prompts (judge CLIP ViT-L/14):")
    print(f"  {RUNGS[0]['name']} raw              {clip_raw:.2f}   ({LAT[0]:.2f} s)")
    print(f"  {RUNGS[0]['name']} + safe polish    {clip_pol:.2f}   (polish changed {changed} pictures)")
    print(f"  {RUNGS[1]['name']} raw              {clip_r1:.2f}   ({LAT[1]:.2f} s)")

## 14 · Examples: when the cascade helped, and when everything failed (for the failure analysis)

In [ ]:
def show(pairs, title, path):
    if not pairs:
        print("No examples for:", title)
        return
    fig, axes = plt.subplots(len(pairs), 2, figsize=(7, 3.4 * len(pairs)))
    axes = np.array(axes).reshape(len(pairs), 2)
    for (a, b, ta, tb, prompt), row in zip(pairs, axes):
        for ax, im, t in zip(row, (a, b), (ta, tb)):
            ax.imshow(im)
            ax.set_title(textwrap.fill(t, 45), fontsize=7)
            ax.axis("off")
        row[0].set_ylabel(textwrap.fill(prompt, 30), fontsize=7)
        row[0].axis("on")
        row[0].set_xticks([])
        row[0].set_yticks([])
    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(path, dpi=110)
    plt.show()

casc = results["cascade from R0"]
rescued = casc[(casc.final > 0) & (casc.source == "geneval") & (casc.judge_struct == True)].head(4)
pairs = []
for _, rr in rescued.iterrows():
    first, last = T.loc[(rr.pid, 0)], T.loc[(rr.pid, int(rr.final))]
    pairs.append((Image.open(IMG_DIR / f"{rr.pid}_{RUNGS[0]['name']}.jpg"), Image.open(IMG_DIR / f"{rr.pid}_{RUNGS[int(rr.final)]['name']}.jpg"),
                  f"{RUNGS[0]['name']}: gate said FAIL: {first['gate_reasons'] or 'low match'}",
                  f"{RUNGS[int(rr.final)]['name']}: gate PASS, judge PASS", info.loc[rr.pid, "prompt"]))
show(pairs, "Cascade rescues: cheap try failed, a later level fixed it", FIG / "cascade_rescues.png")

fails = casc[(casc.source == "geneval") & (casc.judge_struct == False)].head(3)
pairs = []
for _, rr in fails.iterrows():
    last = T.loc[(rr.pid, int(rr.final))]
    pairs.append((Image.open(IMG_DIR / f"{rr.pid}_{RUNGS[0]['name']}.jpg"), Image.open(IMG_DIR / f"{rr.pid}_{RUNGS[int(rr.final)]['name']}.jpg"),
                  f"first try ({RUNGS[0]['name']})", f"final ({RUNGS[int(rr.final)]['name']}): judge says {last['judge_reasons']}",
                  info.loc[rr.pid, "prompt"]))
show(pairs, "Failure cases: still wrong after the cascade", FIG / "failure_cases.png")

# Ours vs the original on the same prompt: where each one wins (both directions, for honesty)
if TEACHER_NAME in results:
    ours = results["router + cascade"].set_index("pid")
    orig = results[TEACHER_NAME].set_index("pid")
    common = [pid for pid in ours.index if pid in orig.index and ours.loc[pid, "source"] == "geneval"]
    wins = [pid for pid in common if ours.loc[pid, "judge_struct"] == True and orig.loc[pid, "judge_struct"] == False]
    losses = [pid for pid in common if ours.loc[pid, "judge_struct"] == False and orig.loc[pid, "judge_struct"] == True]
    print(f"On {len(common)} GenEval test prompts: ours right / original wrong = {len(wins)}; "
          f"original right / ours wrong = {len(losses)}")
    pairs = []
    for pid in wins[:3] + losses[:3]:
        fr = int(ours.loc[pid, "final"])
        pairs.append((Image.open(IMG_DIR / f"{pid}_TEACHER.jpg"), Image.open(IMG_DIR / f"{pid}_{RUNGS[fr]['name']}.jpg"),
                      f"ORIGINAL, 20 steps: judge {'PASS' if orig.loc[pid, 'judge_struct'] == True else 'FAIL'}",
                      f"OURS ({RUNGS[fr]['name']}): judge {'PASS' if ours.loc[pid, 'judge_struct'] == True else 'FAIL'}",
                      info.loc[pid, "prompt"]))
    show(pairs, "Ours vs the original on the same prompt", FIG / "ours_vs_original.png")

## 15 · Save everything for the app and the write-up

In [ ]:
CASCADE_CONFIG = {
    "model": MODEL_REPO, "te_quant": TE_QUANT, "rungs": RUNGS, "rung_latency_s": RUNG_LAT, "gate_latency_s": GATE_LAT,
    "gate": {"clip_model": GATE_CLIP, "detector": "maskrcnn", "clip_min": CLIP_MIN,
             "visual_unfixable_flags": sorted(pcv.UNFIXABLE), "det_threshold": gate.det_threshold,
             "count_threshold": gate.count_threshold},
    "router": ROUTER_JSON, "mode": MODE_TAG,
}
json.dump(CASCADE_CONFIG, open(SHARED / "cascade_config.json", "w"), indent=1)
shutil.make_archive(str(OUT / f"test_images_{MODE_TAG}"), "zip", IMG_DIR)
lines = ["# Notebook 02b results: router + quality-gated cascade", "",
         f"Mode: {MODE_TAG}. Test prompts: {int(te.sum())} ({len(test_geneval)} official GenEval + {len(coco_test)} COCO val). "
         f"Scores from the independent judge (Mask2Former Swin-S + CLIP ViT-L/14), GenEval-style, not official GenEval.", "",
         "## Checker validation on real COCO photos", "", val_table.to_markdown(), "",
         "## Speed", "", speed.round(3).to_markdown(), "",
         "## Strategies on held-out test prompts", "", S.round(3).to_markdown(), "",
         "## GenEval-style accuracy per task", "", per_task.round(3).to_markdown(), "",
         f"Router: CV accuracy on train {cv.mean():.1%}; test accuracy {np.mean(pred[te] == y[te]):.1%} "
         f"(majority baseline {np.mean(y[te] == majority):.1%}).", "",
         "## Gate vs judge agreement", "", agreement.to_markdown(), ""]
(OUT / f"02b_summary_{MODE_TAG}.md").write_text("\n".join(lines))
print("Saved: cascade_config.json (for the app) to", SHARED, "| summary, CSVs, figures, images zip to", OUT)

## 16 · How to read this, and honest limitations

- **The headline is the strategies table.** It answers "does adaptive generation actually help?". Compare **router + cascade** with the best fixed level: similar judge accuracy at a lower average time is a win. If the router doesn't beat the fixed levels, **say so**: the cascade still adapts to the actual picture.
- **The "vs original" claim comes from the ORIGINAL teacher row:** same prompts, same judge. Only claim "better than the original" if our accuracy is higher there. Otherwise the honest claim is "about as good, N× faster". The "ours vs original" gallery shows wins in **both** directions.
- **Counting is the weakest check.** On cluttered real photos, both checkers confirm only about half of true exact counts (section 6). Read counting scores with care.
- **GenEval-style ≠ official GenEval:** different detector implementation, our own re-implementation of the rules, and QUICK mode uses a subset.
- **Only the 80 COCO object types can be checked structurally.** Other prompts rely on CLIP and the visual check.
- **New seed per level:** a failed composition usually needs a *new* sample, not just more steps on the same one. That's why levels change the seed.
- **Latency is GPU time on a T4.** The live app adds network time, plus packing for the phone (notebook 02a).